In [1]:
from bertviz import model_view, head_view
import torch
import torch.nn.functional as F
import tiktoken
import os
import json

In [2]:
with open('moe_config.json', 'r') as f:
    config = json.load(f)

# Hyperparameters
vocab_size = config.get("vocab_size", 100)
d_model = config.get("d_model", 128)
num_heads = config.get("num_heads", 4)
d_ff = config.get("d_ff", 256)
num_layers = config.get("num_layers", 2)
num_experts = config.get("num_experts", 4)
batch_size = config.get("batch_size", 8)

block_size = config.get("block_size", 512)

print(config)

{'vocab_size': 50304, 'd_model': 384, 'd_ff': 1024, 'num_layers': 16, 'num_experts': 16, 'block_size': 2048, 'batch_size': 4, 'max_iters': 900000, 'num_heads': 16, 'log_interval': 10000, 'eval_iters': 100, 'gradient_acc_steps': 4, 'lr': 5e-05, 'resume_training': True, 'load_ckpt': './model_ckpt_3_31_26/ckpt_dist_800000.pt', 'ckpt_num': 800000}


In [3]:
from model_moe import MoETransformer

In [5]:
model_path="model_ckpt\\saved_models_ckpt_dist_900000.pt"
model = MoETransformer(vocab_size, d_model, num_heads, 
                        d_ff, num_layers, num_experts, 
                            max_seq_len=block_size, top_k=2)
model.load_state_dict(
    torch.load(model_path, weights_only=True, map_location="cpu"),
    strict=True
)

<All keys matched successfully>

In [6]:
tk = tiktoken.encoding_for_model("gpt-2")
tokens = tk.encode_ordinary("The Cold War had a significant impact on American economy but it helped shape American dominance")


In [7]:
tk.n_vocab

50257

In [8]:
tokens

[464,
 10250,
 1810,
 550,
 257,
 2383,
 2928,
 319,
 1605,
 3773,
 475,
 340,
 4193,
 5485,
 1605,
 18648]

In [9]:
input_tokens = torch.tensor(
        tokens,
        dtype=torch.long,
        device="cpu"
    )[None, ...]

In [10]:
attn_weights_all_layers = []

def make_hook(layer_idx):
    def hook(module, input, output):
        # input[0] is x: (B, T, d_model) — the input to MultiHeadAttention.forward()
        x = input[0]
        B, T, _ = x.shape

        with torch.no_grad():
            # Reuse the exact same projections the model used
            qkv = module.c_attn(x)
            q, k, _ = qkv.split(module.d_model, dim=2)  # ignore v, not needed for weights

            # Reshape exactly as the model does
            q = q.view(B, T, module.num_heads, module.d_k).transpose(1, 2)  # (B, H, T, d_k)
            k = k.view(B, T, module.num_heads, module.d_k).transpose(1, 2)  # (B, H, T, d_k)

            # Manual scaled dot-product — identical math to SDPA
            scale = module.d_k ** -0.5
            scores = torch.matmul(q, k.transpose(-2, -1)) * scale           # (B, H, T, T)

            # Causal mask — same as is_causal=True in SDPA
            causal_mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
            scores = scores.masked_fill(~causal_mask, float("-inf"))

            weights = F.softmax(scores, dim=-1)                              # (B, H, T, T)

        attn_weights_all_layers.append((layer_idx, weights.detach().cpu()))
    return hook

In [11]:

attn_weights_all_layers.clear()
# Register a hook on each layer's MultiHeadAttention module
hooks = []
for i, block in enumerate(model.blocks):
    h = block.attn.register_forward_hook(make_hook(i))
    hooks.append(h)

with torch.no_grad():
    logits, _ = model(input_tokens)

for h in hooks:
    h.remove()

# attn_weights_all_layers is now a list of (layer_idx, tensor of shape (B, H, T, T))
layer_idx, weights = attn_weights_all_layers[0]
print(weights.shape)  # (B, num_heads, seq_len, seq_len)

torch.Size([1, 16, 16, 16])


C:\Users\bharg\anaconda3\Lib\contextlib.py:109: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


In [12]:
batch_idx = 0
attentions = tuple(
    w[batch_idx].unsqueeze(0)          # (1, num_heads, T, T)
    for _, w in attn_weights_all_layers
)

# Decode token strings for labels

tokens = [tk.decode([t]) for t in input_tokens[batch_idx].tolist()]

In [13]:
attentions[0].shape

torch.Size([1, 16, 16, 16])

In [14]:
attentions[0][0][0]

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.4118e-02, 9.8588e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.3479e-02, 9.3575e-01, 5.0769e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.3470e-02, 9.3330e-01, 5.0586e-02, 2.6472e-03, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.3135e-02, 9.1187e-01, 4.9281e-02, 2.5713e-03, 2.3142e-02, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000

In [15]:
head_view(attentions, tokens)

<IPython.core.display.Javascript object>

In [16]:
model_view(attentions, tokens, include_layers=list(range(0, num_layers)))

<IPython.core.display.Javascript object>